In [17]:
import pandas as pd
dftrain = pd.read_csv("../train_data_2.csv")
dftest = pd.read_csv("../test_data_2.csv")
# Trouver les colonnes communes
common_cols = dftrain.columns.intersection(dftest.columns)

# Garder uniquement les colonnes communes
dftrain = dftrain[common_cols].dropna()
dftest = dftest[common_cols].dropna()

print("Colonnes communes conservées :")
print(common_cols)

Colonnes communes conservées :
Index(['name', '64-bit execution via heavens gate', '64bits', 'PEB access',
       'accept command line arguments', 'access the Windows event log',
       'act as TCP client', 'allocate RW memory', 'allocate RWX memory',
       'allocate memory',
       ...
       'pminuw', 'vpaddusw', 'phaddw', 'pminud', 'pblendvb', 'vcmpps', 'rcpss',
       'cmpeqpd', 'vpsubusw', 'cmpnltsd'],
      dtype='object', length=980)


In [23]:
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.multioutput import MultiOutputClassifier

# Sélection des features et labels
features = [
    "num_nodes", "num_edges", "avg_degree", "num_scc", "avg_clustering", 
    "avg_degree_centrality", "avg_closeness_centrality", "avg_betweenness_centrality", 
    "num_cycles", "avg_cycle_length"
]

#X = df[features]  # Features
X = dftrain.iloc[:, 454:]
y = dftrain.iloc[:, 1:454]  # Labels binaires/multiclasses

# Division en train (80%) et test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scale_pos_weights = (y_train.shape[0] - y_train.sum(axis=0)) / (y_train.sum(axis=0) + 1e-6)
# Modèle KNN avec k=5
knn = MultiOutputClassifier(KNeighborsClassifier(n_neighbors=5))
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

# Calcul du F1-score macro moyen

In [24]:
f1_macro = f1_score(y_test, y_pred, average='macro')
print(f"F1-score macro: {f1_macro:.4f}")

F1-score macro: 0.0214


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [25]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import StandardScaler

# Sélection des features et labels
features = [
    "num_nodes", "num_edges", "avg_degree", "num_scc", "avg_clustering", 
    "avg_degree_centrality", "avg_closeness_centrality", "avg_betweenness_centrality", 
    "num_cycles", "avg_cycle_length"
]

# Exemple : combiner les features globales et les occurrences d'instructions d'assembleur
# Ici, on suppose que dftrain contient à la fois les features globales et les 500 instructions
X = dftrain.iloc[:, 454:]  # Ajuste les indices selon ton DataFrame
y = dftrain.iloc[:, 1:454]

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Recherche du meilleur k avec GridSearch sur un KNN simple label par label
param_grid = {
    'estimator__n_neighbors': [3, 5, 7, 10],
    'estimator__weights': ['uniform', 'distance'],
    'estimator__metric': ['euclidean', 'manhattan', 'cosine']
}

knn = MultiOutputClassifier(KNeighborsClassifier())
grid = GridSearchCV(knn, param_grid, scoring='f1_macro', cv=3, n_jobs=-1)
grid.fit(X_train_scaled, y_train)

print("Meilleurs hyperparamètres :", grid.best_params_)

# Utiliser le meilleur modèle
best_knn = grid.best_estimator_
y_pred = best_knn.predict(X_test_scaled)

f1_macro_per_label = [f1_score(y_test.iloc[:, i], y_pred[:, i], average='weighted', zero_division=0)
                      for i in range(y_test.shape[1])]
f1_macro = np.mean(f1_macro_per_label)
print(f"F1-score macro: {f1_macro:.4f}")

/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/l

KeyboardInterrupt: 

In [ ]:
import joblib

# Save the model to a file
joblib.dump(knn, 'knn_model.pkl')

In [ ]:
# Lecture du fichier de test
df_to_predict = dftest
df_to_predict

In [ ]:
knn = joblib.load('knn_model.pkl')
# Extraire la colonne "name" et la conserver

names = df_to_predict["name"]

# Supprimer la colonne "name" pour conserver uniquement les features
df_features = df_to_predict.drop("name", axis=1)

# Assurez-vous que les features correspondent à celles utilisées lors de l'entraînement
X_to_predict = df_features[X.columns]

# Effectuer la prédiction avec le modèle multi-output
predictions = knn.predict(X_to_predict)

# Créer un DataFrame pour les prédictions avec les mêmes noms de colonnes que pour y_train
predictions_df = pd.DataFrame(predictions, columns=y_train.columns)

# Réinsérer la colonne "name" en première position
predictions_df.insert(0, "name", names.values)

# Exporter les prédictions dans un fichier CSV
predictions_df.to_csv("predictions_knn.csv", index=False)

In [ ]:
predictions_df